In [1]:
import os
import json
import boto3
import torch
import tempfile
import numpy as np
import pandas as pd
from copy import deepcopy
from typing import List, Tuple, Dict, Optional, Any, Set
from dataclasses import dataclass, field
from torch.utils.data import Dataset, DataLoader, random_split
from torch import nn

In [2]:
try:
    from transformer_lens import HookedTransformer
except ImportError:
    HookedTransformer = Any

2026-03-07 20:52:49.380358: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-07 20:52:49.424593: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-07 20:52:50.307240: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
/home/ubuntu/minicon

In [21]:
@dataclass
class ExperimentConfig:
    """Central configuration for the fine-tuning experiments."""
    fine_tune_dataset: str = "data/stereoset/fine-tune-sft/sft_bias_mitigation.jsonl"
    train_file_path: str = "data/stereoset/splits/gender_train.json"

    s3_bucket: str = "modelsfinetuned"
    s3_prefix: str = "gpt2-xl-finetuned"
    checkpoint_dir: str = "checkpoints"

    batch_size: int = 4
    learning_rate: float = 1e-5
    num_epochs: int = 10
    patience: int = 5
    max_token_length: int = 48

    # loss_type: str = "kl_divergence"
    kl_weight: float = 0.05

    percentiles: List[int] = field(default_factory=lambda: [100])
    bias_type: str = 'gender'

    experiment_type: str = "full"

    def __post_init__(self):
        os.makedirs(self.checkpoint_dir, exist_ok=True)

## Fine-tuning code

In [18]:
def save_checkpoint(model, s3_client=None, s3_bucket=None, s3_key=None):
    """Saves locally and optionally uploads to S3."""

    with tempfile.NamedTemporaryFile(delete=False, suffix=".pt") as tmp:
        torch.save(model.state_dict(), tmp.name)

        try:
            s3_client.upload_file(tmp.name, s3_bucket, s3_key)
            print(f"--> Uploaded to s3://{s3_bucket}/{s3_key}")
        except Exception as e:
            print(f"!! Failed to upload to s3://{s3_bucket}/{s3_key}: {e}")
        finally:
            os.remove(tmp.name)

def identify_top_impact_heads(
    df_impact: pd.DataFrame,
    df_probs: pd.DataFrame,
    df_impact_analysis: pd.DataFrame,
    percentile: float
) -> Tuple[pd.Series, List[str]]:

    head_df = df_impact_analysis[
        (df_impact_analysis['Model_Preference'] == 'stereotype') &
        (df_impact_analysis['Component'].str.startswith('Head'))
    ].copy()

    head_df['Head_ID'] = head_df['Layer'].astype(str) + "_" + head_df['Component']

    mean_impact = head_df.groupby('Head_ID')['Accumulated_Impact'].mean()

    threshold = mean_impact.quantile(1 - (percentile / 100))
    top_heads = mean_impact[mean_impact >= threshold].sort_values(ascending=False)

    target_ids = head_df['ID'].unique().tolist()

    return top_heads, target_ids

def identify_top_mlp_impact(
    df_impact: pd.DataFrame,
    df_probs: pd.DataFrame,
    df_impact_analysis: pd.DataFrame,
    percentile: float,
):

    mlp_df = df_impact_analysis[
        (df_impact_analysis['Model_Preference'] == 'stereotype') &
        (df_impact_analysis['Component'].str.startswith('MLP'))
    ].copy()

    mlp_df['MLP_ID'] = mlp_df['Layer'].astype(str) + "_" + mlp_df['Component']

    mean_impact = mlp_df.groupby('MLP_ID')['Accumulated_Impact'].mean()

    threshold = mean_impact.quantile(1 - (percentile / 100))
    top_mlps = mean_impact[mean_impact >= threshold].sort_values(ascending=False)

    target_ids = mlp_df['ID'].unique().tolist()

    return top_mlps, target_ids

def identify_top_mlp_probability(
    df_impact: pd.DataFrame,
    df_probs: pd.DataFrame,
    df_impact_analysis : pd.DataFrame,
    percentile: float,
) -> Tuple[pd.Series, List[str]]:

    mlp_df = df_impact_analysis[
        (df_impact_analysis['Model_Preference'] == 'stereotype') &
        (df_impact_analysis['Component'].str.startswith('MLP'))
    ].copy()

    mlp_df['MLP_ID'] = mlp_df['Layer'].astype(str) + "_" + mlp_df['Component']

    target_ids = mlp_df['ID'].unique().tolist()

    if not target_ids:
        return pd.Series(dtype=float), []


    idx_layer_last_token = df_probs.groupby(['ID', 'Type', 'Layer'])['Token_Position'].transform('max') == df_probs['Token_Position']

    stereo_probs = df_probs[
        (df_probs['ID'].isin(target_ids)) &
        (df_probs['Type'] == 'stereotype') &
        (idx_layer_last_token)
    ].copy()

    stereo_probs = stereo_probs.drop_duplicates(subset=['ID', 'Layer'])

    mean_layer_probs = stereo_probs.groupby('Layer')['Layer_Accumulated_Prob'].mean()

    mean_layer_probs.index = mean_layer_probs.index.astype(str) + "_MLP"

    threshold = mean_layer_probs.quantile(1 - (percentile / 100))
    top_mlps = mean_layer_probs[mean_layer_probs >= threshold].sort_values(ascending=False)

    return top_mlps, target_ids



def identify_top_layers_attn_impact_mlp(
    df_impact: pd.DataFrame,
    df_probs: pd.DataFrame,
    df_impact_analysis : pd.DataFrame,
    percentile: float,
) -> Tuple[pd.Series, List[str]]:

    head_df = df_impact_analysis[
        (df_impact_analysis['Model_Preference'] == 'stereotype') &
        (df_impact_analysis['Component'].str.startswith('Head'))
    ].copy()

    head_df['Head_ID'] = head_df['Layer'].astype(str) + "_" + head_df['Component']
    mean_head_impact = head_df.groupby('Head_ID')['Accumulated_Impact'].mean()

    threshold = mean_head_impact.quantile(1 - (percentile / 100))
    top_heads = mean_head_impact[mean_head_impact >= threshold]

    top_layers = list(set([int(head_id.split('_')[0]) for head_id in top_heads.index]))

    mlp_df = df_impact_analysis[
        (df_impact_analysis['Model_Preference'] == 'stereotype') &
        (df_impact_analysis['Component'] == 'MLP') &
        (df_impact_analysis['Layer'].isin(top_layers))
    ].copy()

    mlp_df['Component_ID'] = mlp_df['Layer'].astype(str) + "_" + mlp_df['Component']
    mean_mlp_impact = mlp_df.groupby('Component_ID')['Accumulated_Impact'].mean()

    combined_impacts = pd.concat([top_heads, mean_mlp_impact]).sort_values(ascending=False)

    target_ids = df_impact_analysis[df_impact_analysis['Model_Preference'] == 'stereotype']['ID'].unique().tolist()

    return combined_impacts, target_ids

def df_impact_analysis_selection(
    df_impact: pd.DataFrame,
    df_probs: pd.DataFrame
):
    df_impact_analysis = df_impact.copy()

    max_layer = df_impact_analysis['Layer'].max()
    idx_last_token = df_probs.groupby(['ID', 'Type'])['Token_Position'].transform('max') == df_probs['Token_Position']

    final_probs = df_probs[
        (df_probs['Layer'] == max_layer) &
        (idx_last_token)
    ].copy()
    final_probs = final_probs.drop_duplicates(subset=['ID', 'Type'])

    prob_pivot = final_probs.pivot(index='ID', columns='Type', values='Layer_Accumulated_Prob')

    conditions = [
        (prob_pivot['stereotype'] > prob_pivot['anti-stereotype']) & (prob_pivot['stereotype'] > prob_pivot['unrelated']),
        (prob_pivot['anti-stereotype'] > prob_pivot['stereotype']) & (prob_pivot['anti-stereotype'] > prob_pivot['unrelated']),
        (prob_pivot['unrelated'] > prob_pivot['stereotype']) & (prob_pivot['unrelated'] > prob_pivot['anti-stereotype'])
    ]
    prob_pivot['Winner_Type'] = np.select(conditions, ['stereotype', 'anti-stereotype', 'unrelated'], default='tie')
    df_impact_analysis['Model_Preference'] = df_impact_analysis['ID'].map(prob_pivot['Winner_Type'].to_dict())

    return df_impact_analysis


## old code

In [23]:
import os
import json
import torch
from torch.utils.data import Dataset
from typing import Optional, List

class DebiasingDataset(Dataset):
    def __init__(self,
                 json_path: str,
                 tokenizer,
                 target_ids: Optional[List[str]] = None,
                 max_length: int = 128):
        self.tokenizer = tokenizer
        self.data = []  # Will now store tuples of (prompt, completion)
        self.max_length = max_length

        # Ensure tokenizer padding side is correct for GPT-2 training
        if self.tokenizer.padding_side != 'right':
            self.tokenizer.padding_side = 'right'

        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        if not os.path.exists(json_path):
            raise FileNotFoundError(f"Dataset not found at {json_path}")

        # Robust Loading: Handle both JSON and JSONL
        with open(json_path, 'r', encoding='utf-8') as f:
            try:
                raw_data = json.load(f)
            except json.JSONDecodeError:
                f.seek(0)
                raw_data = [json.loads(line) for line in f]

        target_ids_set = set(target_ids) if target_ids else None

        print(f"Loading data from {json_path}...")

        for item in raw_data:
            # ID Filtering Logic (if IDs are included in the generated JSONL)
            if target_ids_set is not None and 'id' in item:
                if str(item['id']) not in target_ids_set:
                    continue

            # --- New Structure Support (Prompt / Completion) ---
            if 'prompt' in item and 'completion' in item:
                prompt_text = item['prompt']
                # Append EOS token to the completion so the model learns to stop generating
                completion_text = item['completion']

                if prompt_text and completion_text:
                    self.data.append((prompt_text, completion_text))

            # --- Fallback Support (Just in case older formats slip in) ---
            elif 'messages' in item:
                text_input = ""
                for msg in item['messages']:
                    role = msg.get('role', '').capitalize()
                    content = msg.get('content', '')
                    text_input += f"{role}: {content}\n"
                self.data.append((text_input.strip(), self.tokenizer.eos_token))

        print(f"Loaded {len(self.data)} examples for fine-tuning.")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        prompt, completion = self.data[idx]

        # 1. Combine prompt and completion for full sequence tokenization
        full_text = prompt + completion

        # 2. Tokenize the full text
        encoded_full = self.tokenizer(
            full_text,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )

        # 3. Tokenize JUST the prompt to figure out how many tokens it takes up
        # add_special_tokens=False prevents the tokenizer from adding BOS tokens that would skew the count
        encoded_prompt = self.tokenizer(
            prompt,
            truncation=True,
            max_length=self.max_length,
            add_special_tokens=False,
            return_tensors="pt"
        )

        prompt_length = encoded_prompt["input_ids"].shape[1]

        input_ids = encoded_full["input_ids"].squeeze(0)
        attention_mask = encoded_full["attention_mask"].squeeze(0)

        # 4. Create the labels tensor
        labels = input_ids.clone()

        # 5. Mask out the prompt tokens so the model doesn't train on them
        # (Only calculate loss on the completion)
        labels[:prompt_length] = -100

        # 6. Mask out padding tokens
        labels[attention_mask == 0] = -100

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
        }

def get_gradient_mask_hook(mask: torch.Tensor):
    """Creates a hook that applies a binary mask to gradients."""
    def hook(grad):
        return grad * mask
    return hook

def configure_trainable_parameters(
    model: HookedTransformer,
    target_components: List[str],
    condition: str = 'attn'
) -> Tuple[HookedTransformer, int, List[Any]]:
    """
    Freezes the model and selectively unfreezes specific attention heads or MLPs.
    Returns the model, the count of active parameters, and a list of hook handles.
    """
    # 1. Freeze all parameters initially
    if condition != 'full':
        for param in model.parameters():
            param.requires_grad = False

    attn_head_targets_by_layer = {}
    mlp_targets = set()

    # 2. Parse target components
    if condition == 'attn':
        for item in target_components:
            # Example item: "10_Head_5"
            parts = item.split('_')
            if len(parts) >= 3:
                layer_idx, head_idx = int(parts[0]), int(parts[2])
                attn_head_targets_by_layer.setdefault(layer_idx, []).append(head_idx)

    elif condition in ['mlp_impact_only', 'mlp_probability_only']:
        for item in target_components:
            # Example item: "10_MLP"
            parts = item.split('_')
            mlp_targets.add(int(parts[0]))

    elif condition == 'attn_impact_mlp':
        for item in target_components:
            parts = item.split('_')
            if len(parts) >= 3:
                layer_idx, head_idx = int(parts[0]), int(parts[2])
                attn_head_targets_by_layer.setdefault(layer_idx, []).append(head_idx)
            else:
                mlp_targets.add(int(parts[0]))


    active_params_count = 0
    total_params = 0
    hook_handles = []

    n_heads = model.cfg.n_heads

    # 3. Apply Hooks and Unfreeze selectively
    for name, param in model.named_parameters():
        total_params += param.numel()

        parts = name.split(".")
        # TransformerLens block parameters start with "blocks.X."
        if len(parts) < 3 or parts[0] != "blocks":
            continue

        try:
            layer_idx = int(parts[1])
        except ValueError:
            continue

        # --- Attention unfreezing ---
        if (condition == 'attn' or condition == 'attn_impact_mlp') and layer_idx in attn_head_targets_by_layer and "attn" in name:
            active_heads = attn_head_targets_by_layer[layer_idx]

            # TransformerLens convention: W_Q, W_K, W_V are [n_heads, d_model, d_head]
            # W_O is [n_heads, d_head, d_model], b_Q/K/V are [n_heads, d_head]
            # We target parameters where the first dimension is n_heads.
            if param.shape[0] == n_heads:
                param.requires_grad = True
                mask = torch.zeros_like(param)

                # The ellipsis (...) broadcasts the 1.0 to all remaining dimensions
                # (e.g., d_model and d_head) for the specified active_heads.
                mask[active_heads, ...] = 1.0

                handle = param.register_hook(get_gradient_mask_hook(mask))
                hook_handles.append(handle)

                # Accurately count only the unmasked parameters
                params_per_head = param.numel() // n_heads
                active_params_count += params_per_head * len(active_heads)

        # --- MLP unfreezing ---
        elif condition in ['mlp_impact_only', 'attn_impact_mlp', 'mlp_probability_only'] and layer_idx in mlp_targets and "mlp" in name:
            param.requires_grad = True
            active_params_count += param.numel()

    print(f"\n--- Unfreezing Summary ({condition}) ---")

    if condition == 'attn':
        print(f"Targeted Layers (Attn): {list(attn_head_targets_by_layer)}")
    elif condition in ['mlp_impact_only', 'mlp_probability_only']:
        print(f"Targeted Layers (MLP): {list(mlp_targets)}")
    elif condition == 'attn_impact_mlp':
        print(f"Targeted Layers (Attn): {list(attn_head_targets_by_layer)}")
        print(f"Targeted Layers (MLP): {list(mlp_targets)}")

    if condition == 'full':
        active_params_count = total_params
    print(f"Active parameters: {active_params_count:,} / {total_params:,}\n")

    return model, active_params_count, hook_handles

def run_training(
    model: HookedTransformer,
    ref_model: HookedTransformer,
    train_loader: DataLoader,
    val_loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    config: ExperimentConfig,
    run_id: str
):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)

    ref_model.to(device)
    ref_model.eval()

    best_val_loss = float('inf')
    patience_counter = 0

    s3_client = None
    if config.s3_bucket:
         s3_client = boto3.client('s3',
                             aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
                             aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"])

    print(f"Starting training run: {run_id}")

    loss_fct = nn.CrossEntropyLoss(ignore_index=-100)

    for epoch in range(config.num_epochs):
        model.train()
        total_train_loss = 0
        total_sft_loss = 0
        total_kl_loss = 0

        for batch in train_loader:
            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad()

            logits = model(input_ids)

            with torch.no_grad():
                ref_logits = ref_model(input_ids)

            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            sft_loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))

            pad_mask = (input_ids != tokenizer.pad_token_id).view(-1)

            flat_logits = logits.view(-1, logits.size(-1))[pad_mask]
            flat_ref_logits = ref_logits.view(-1, ref_logits.size(-1))[pad_mask]

            log_probs = torch.nn.functional.log_softmax(flat_logits, dim=-1)
            ref_probs = torch.nn.functional.softmax(flat_ref_logits, dim=-1)

            kl_loss = torch.nn.functional.kl_div(log_probs, ref_probs, reduction='batchmean')
            loss = sft_loss + (config.kl_weight * kl_loss)

            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()
            total_sft_loss += sft_loss.item()
            total_kl_loss += kl_loss.item()

        avg_train_loss = total_train_loss / len(train_loader)
        avg_sft_loss = total_sft_loss / len(train_loader)
        avg_kl_loss = total_kl_loss / len(train_loader)

        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch["input_ids"].to(device)
                labels = batch["labels"].to(device)

                logits = model(input_ids)
                shift_logits = logits[..., :-1, :].contiguous()
                shift_labels = labels[..., 1:].contiguous()

                val_loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
                total_val_loss += val_loss.item()

        avg_val_loss = total_val_loss / len(val_loader)

        print(f"Epoch {epoch+1} | Train: {avg_train_loss:.4f} | Val: {avg_val_loss:.4f}")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0

            filename = f"best_model_{run_id}_epoch_{epoch}.pt"
            s3_key = f"{config.s3_prefix}/{filename}"

            print(f"--> Improvement detected. Saving...")

            save_checkpoint(model, s3_client, config.s3_bucket, s3_key)
        else:
            patience_counter += 1
            if patience_counter >= config.patience:
                print("--> Early stopping triggered.")
                break

    return best_val_loss

def run_experiments(
    model: HookedTransformer,
    tokenizer,
    df_impact: pd.DataFrame,
    df_probability_info: pd.DataFrame,
    config: ExperimentConfig
):
    original_state_dict = deepcopy(model.state_dict())

    experiment_results = {}

    df_impact_analysis = df_impact_analysis_selection(df_impact, df_probability_info)

    for percentile in config.percentiles:
        print(f"\n{'='*40}\nRunning Experiment: Top {percentile}% Targets\n{'='*40}")

        target_ids = []

        top_heads = pd.Series()
        combined_impacts = pd.Series()
        top_mlps = pd.Series()

        if config.experiment_type == 'attn':
            top_heads, target_ids = identify_top_impact_heads(
                df_impact, df_probability_info, df_impact_analysis, percentile
            )

        if config.experiment_type == 'attn_impact_mlp':
            combined_impacts, target_ids = identify_top_layers_attn_impact_mlp(
                df_impact, df_probability_info, df_impact_analysis, percentile
            )

        if config.experiment_type == 'mlp_impact_only':
            top_mlps, target_ids = identify_top_mlp_impact(
                        df_impact, df_probability_info, df_impact_analysis, percentile
                    )

        if config.experiment_type == 'mlp_probability_only':
            top_mlps, target_ids = identify_top_mlp_probability(
                df_impact, df_probability_info, df_impact_analysis, percentile
            )

        if config.experiment_type == 'full':
            target_ids = df_impact_analysis[df_impact_analysis['Model_Preference'] == 'stereotype']['ID'].unique().tolist()


        if len(target_ids) == 0:
            print("No target examples found for this percentile. Skipping.")
            continue

        dataset = DebiasingDataset(
            config.fine_tune_dataset,
            tokenizer,
            target_ids=target_ids,
            max_length=config.max_token_length
        )

        if len(dataset) == 0:
             print("Dataset is empty after filtering/loading. Skipping experiment.")
             continue

        train_size = int(0.8 * len(dataset))
        val_size = len(dataset) - train_size
        train_set, val_set = random_split(
            dataset, [train_size, val_size],
            generator=torch.Generator().manual_seed(42)
        )

        train_loader = DataLoader(train_set, batch_size=config.batch_size, shuffle=True)
        val_loader = DataLoader(val_set, batch_size=config.batch_size, shuffle=False)

        target_components = []
        if config.experiment_type == 'attn':
            target_components = top_heads.index.tolist()
        elif config.experiment_type in ['mlp_impact_only', 'mlp_probability_only']:
            target_components = top_mlps.index.tolist()
        elif config.experiment_type == 'attn_impact_mlp':
            target_components = combined_impacts.index.tolist()

        # print(target_components)

        model, num_params, hook_handles = configure_trainable_parameters(
            model,
            target_components=target_components,
            condition=config.experiment_type
        )

        optimizer = torch.optim.AdamW(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=config.learning_rate,
            weight_decay=0.0
        )

        ref_model = HookedTransformer.from_pretrained("gpt2-xl")
        for param in ref_model.parameters():
            param.requires_grad = False

        best_loss = run_training(
            model,
            ref_model,
            train_loader,
            val_loader,
            optimizer,
            config, run_id=f"{config.experiment_type}_{percentile}"
        )

        experiment_results[percentile] = {
            "best_val_loss": best_loss,
            "trainable_params": num_params
        }

        print("Cleaning up hooks and resetting weights...")

        for handle in hook_handles:
            handle.remove()
        hook_handles.clear()

        model.load_state_dict(original_state_dict)
        for param in model.parameters():
            param.requires_grad = True

    return experiment_results


## call

In [10]:
model = HookedTransformer.from_pretrained("gpt2-xl")

tokenizer = model.tokenizer

df_impact = pd.read_csv("outputs/gpt2-xl/dev_tests/accumulated_impact_gender_train.csv")
df_probs = pd.read_csv("outputs/gpt2-xl/dev_tests/out_DLA_gender_train.csv")

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model gpt2-xl into HookedTransformer


In [24]:
config = ExperimentConfig()
results = run_experiments(model,
                          tokenizer,
                          df_impact,
                          df_probs,
                          config)


Running Experiment: Top 100% Targets
Loading data from data/stereoset/fine-tune-sft/sft_bias_mitigation.jsonl...
Loaded 393 examples for fine-tuning.

--- Unfreezing Summary (full) ---
Active parameters: 1,637,762,257 / 1,637,762,257

Loaded pretrained model gpt2-xl into HookedTransformer
Moving model to device:  cpu
Moving model to device:  cpu
Starting training run: full_100
Epoch 1 | Train: 0.6954 | Val: 0.3347
--> Improvement detected. Saving...
--> Uploaded to s3://modelsfinetuned/gpt2-xl-finetuned/best_model_full_100_epoch_0.pt
Epoch 2 | Train: 0.3114 | Val: 0.3235
--> Improvement detected. Saving...
--> Uploaded to s3://modelsfinetuned/gpt2-xl-finetuned/best_model_full_100_epoch_1.pt
Epoch 3 | Train: 0.2545 | Val: 0.3788
Epoch 4 | Train: 0.1750 | Val: 0.4338
Epoch 5 | Train: 0.1389 | Val: 0.4670
Epoch 6 | Train: 0.1181 | Val: 0.4862
Epoch 7 | Train: 0.1092 | Val: 0.4975
--> Early stopping triggered.
Cleaning up hooks and resetting weights...


In [9]:
import json
import pandas as pd
from typing import List

def generate_parameter_stats_json(
    model,
    df_impact: pd.DataFrame,
    df_probs: pd.DataFrame,
    percentiles: List[float],
    conditions: List[str]
) -> str:
    """
    Calculates active parameters and their percentages for given percentiles and conditions,
    returning the results as a formatted JSON string.
    """
    results = []

    # Calculate the total number of parameters in the model once
    total_model_params = sum(p.numel() for p in model.parameters())

    for condition in conditions:
        for percentile in percentiles:
            # 1. Identify target components based on the specific condition
            if condition == 'attn':
                top_components, _ = identify_top_impact_heads(df_impact, df_probs, percentile)
            elif condition == 'mlp_impact_only':
                top_components, _ = identify_top_mlp_impact(df_impact, df_probs, percentile)
            elif condition == 'mlp_probability_only':
                top_components, _ = identify_top_mlp_probability(df_impact, df_probs, percentile)
            elif condition == 'attn_impact_mlp':
                top_components, _ = identify_top_layers_attn_impact_mlp(df_impact, df_probs, percentile)
            else:
                raise ValueError(f"Unknown condition: {condition}")

            # Extract the component names from the index of the returned Series
            target_components = top_components.index.tolist() if not top_components.empty else []

            # 2. Configure parameters and get the active count
            _, active_params_count, hook_handles = configure_trainable_parameters(
                model=model,
                target_components=target_components,
                condition=condition
            )

            # Remove hooks immediately after counting so they don't accumulate and cause memory leaks
            for handle in hook_handles:
                handle.remove()

            # 3. Calculate percentage
            percentage = (active_params_count / total_model_params) * 100 if total_model_params > 0 else 0.0

            # 4. Append to results
            results.append({
                "condition": condition,
                "percentile": percentile,
                "active_parameters": active_params_count,
                "total_parameters": total_model_params,
                "percentage_of_full_model": round(percentage, 4)
            })

    # Return the data as a neatly formatted JSON string
    return json.dumps(results, indent=4)

percentiles_to_test = [0.2, 0.5, 0.8, 1.0, 2.0, 5.0, 10.0, 20.0]
conditions_to_test = ['attn', 'mlp_impact_only', 'mlp_probability_only', 'attn_impact_mlp']

json_output = generate_parameter_stats_json(
    model=model,
    df_impact=df_impact,
    df_probs=df_probs,
    percentiles=percentiles_to_test,
    conditions=conditions_to_test
)
print(json_output)


--- Unfreezing Summary (attn) ---
Targeted Layers (Attn): [40, 37, 34]
Active parameters: 1,229,376 / 1,637,762,257


--- Unfreezing Summary (attn) ---
Targeted Layers (Attn): [40, 37, 34, 41, 42, 31]
Active parameters: 2,458,752 / 1,637,762,257


--- Unfreezing Summary (attn) ---
Targeted Layers (Attn): [40, 37, 34, 41, 42, 31, 32, 25, 33]
Active parameters: 4,097,920 / 1,637,762,257


--- Unfreezing Summary (attn) ---
Targeted Layers (Attn): [40, 37, 34, 41, 42, 31, 32, 25, 33, 27]
Active parameters: 4,917,504 / 1,637,762,257


--- Unfreezing Summary (attn) ---
Targeted Layers (Attn): [40, 37, 34, 41, 42, 31, 32, 25, 33, 27, 26, 30, 22, 29, 47, 36, 28]
Active parameters: 9,835,008 / 1,637,762,257


--- Unfreezing Summary (attn) ---
Targeted Layers (Attn): [40, 37, 34, 41, 42, 31, 32, 25, 33, 27, 26, 30, 22, 29, 47, 36, 28, 38, 44, 21, 35, 5, 2, 6, 23, 20, 46, 19, 12, 24, 4, 1, 39, 17, 45]
Active parameters: 24,587,520 / 1,637,762,257


--- Unfreezing Summary (attn) ---
Targeted Laye